# Efficient Browsing

Notebook 12's browse flow is `GET /photos/?page=N`. Two problems compound with scale. **First, offset pagination is unstable and slow.** Page 1000 of a 50 000-photo library runs `LIMIT 30 OFFSET 30 000`, which makes Postgres scan 30 000 rows and discard them — a per-request cost that grows linearly with how deep the user has scrolled, and **renders wrong** the moment a new photo is uploaded (every page shifts down by one). **Second, presigned URLs are regenerated per request.** A page of 30 photos burns 30 `generate_presigned_url` S3 API calls, each ~5–20 ms over the network, on every browse. Even on a fast LAN this is a 600 ms floor per page before any camera or pipeline work happens.

This notebook closes both. We replace offset paging with **keyset pagination** over a composite `(taken_at DESC, photo_id DESC)` cursor, making every page a constant-time index seek. We add a **Redis-backed presigned-URL cache** that returns the same URL for a `(bucket, key)` pair across requests as long as the URL's clock has not expired, averaging URL generation to one round-trip per `(bucket, key)` per TTL window. Together, a browse request at 100K photos goes from a sub-second best-case interaction on warm cache to a worst-case of one DB seek + zero S3 presign calls.

<br>

**The performance budget.** Before any optimizations, the browse handler consists of (a) one SQL query, (b) 30 presigned-URL generations, (c) JSON serialization. On warm cache, (a) stays at ~5 ms, (b) drops to ~0 ms, (c) stays at ~1 ms. The cold path adds 30 S3 presign calls, ≈300 ms. The throughput-critical question is whether your library grows past the point where offset pagination dominates — and at 50K photos you are already there.

---

## Why Keyset, Not Offset

Offset pagination computes "skip then take." Keyset pagination computes "take ones after this point." For an `ORDER BY taken_at DESC, photo_id DESC` ordering, the "next page" predicate is:

```sql
WHERE (taken_at, photo_id) < (:cursor_taken_at, :cursor_photo_id)
ORDER BY taken_at DESC, photo_id DESC
LIMIT :page_size
```

This is a single index seek — Postgres walks a b-tree starting at the cursor position. The cost is **constant** in the depth of the scroll, where offset paging is **linear**. The cursor is opaque to the client (a base64 of `(taken_at, photo_id)`), which prevents clients from browsing to arbitrary page numbers but eliminates the "page 1000 shifts after an upload" failure mode entirely — what you scrolled past earlier is still in the same position.

:::{.callout-caution}
Keyset pagination forbids "jump to page N." The client can only scroll forward and backward from where it currently is. For a personal photo library this is the right trade-off — users scroll the timeline, they do not jump to page 47. If you have a search results UI that exposes page numbers, keep offset paging for search and use keyset for timeline.

:::


## Cursor Encoding

The cursor is two values — the `taken_at` and `photo_id` of the **last item on the current page** — packed into an opaque base64 string. We use base64 because (a) the cursor travels in a query parameter (`?cursor=...`), (b) `taken_at` includes a timestamp with optional fractional seconds and a colon, both of which need URL-safe handling, and (c) opaque cursors survive minor schema changes (e.g. switching to a `uuid` photo_id) without clients having to know.


In [ ]:
import base64
import json
from datetime import datetime


def encode_cursor(taken_at: datetime, photo_id: str) -> str:
    raw = json.dumps({
        "taken_at": taken_at.isoformat(),
        "photo_id": photo_id,
    }).encode("utf-8")
    return base64.urlsafe_b64encode(raw).decode("ascii").rstrip("=")


def decode_cursor(cursor: str) -> tuple[datetime, str]:
    pad = "=" * (-len(cursor) % 4)
    payload = json.loads(base64.urlsafe_b64decode(cursor + pad))
    return datetime.fromisoformat(payload["taken_at"]), payload["photo_id"]


# Round-trip.
ts = datetime(2023, 8, 14, 17, 42, 0)
c = encode_cursor(ts, "p-001")
print(f"cursor: {c}  (length {len(c)})")
back_ts, back_id = decode_cursor(c)
print(f"decoded: {back_ts!r}  id={back_id}  ok={back_ts == ts and back_id == 'p-001'}")


## The Keyset Query

We build the predicate `WHERE (taken_at, photo_id) < (:ts, :id)` in SQLAlchemy using a row-tuple comparison. This works on Postgres directly; on MySQL it would need to be expanded into `(taken_at < :ts) OR (taken_at = :ts AND photo_id < :id)` — but our database is Postgres, and pgvector's host has supported row-value predicates since v8.


In [ ]:
from sqlalchemy import select, tuple_, text
from sqlalchemy.ext.asyncio import AsyncSession


# Stand-in for the photos ORM model. In production this would be a SQLAlchemy
# declarative model defined in the photo app's models module.
class PhotoORM:
    """Shape only — used to build typed queries against the photos table."""
    taken_at = None
    photo_id = None
    sha256 = None


async def fetch_page_keyset(
    session: AsyncSession,
    page_size: int,
    cursor: str | None,
) -> tuple[list[dict], str | None]:
    """Fetch a page of photos using keyset pagination. Returns (rows, next_cursor).
   	next_cursor is None when there are no more rows."""
    stmt = select(PhotoORM).order_by(PhotoORM.taken_at.desc(), PhotoORM.photo_id.desc()).limit(page_size + 1)
    if cursor is not None:
        cursor_ts, cursor_id = decode_cursor(cursor)
        # Row-tuple comparison: Postgres short-circuits via the composite index.
        stmt = stmt.where(
            tuple_(PhotoORM.taken_at, PhotoORM.photo_id) < (cursor_ts, cursor_id)
        )
    # In production: rows = (await session.execute(stmt)).scalars().all()
    # Here we stub: return an empty list so the function compiles + runs.
    rows: list[dict] = []
    if len(rows) <= page_size:
        return rows, None
    page = rows[:page_size]
    last = page[-1]
    return page, encode_cursor(last["taken_at"], last["photo_id"])


## Why `(taken_at, photo_id)` and Not `taken_at` Alone

`taken_at` alone is not unique — two photos taken in the same second are interleaved arbitrarily by the database, and **forgetting one** in the scroll is the failure mode the cursor must prevent (cursor "(2023-08-14 17:42:00, ?)" can land between two photos at that exact timestamp). Adding `photo_id` as a tiebreaker makes `(taken_at, photo_id)` a strict total order, which guarantees every photo appears exactly once across the infinite scroll, regardless of repeated timestamps.

The Postgres composite index `(taken_at DESC, photo_id DESC)` makes the predicate an index seek — the executor reads the index from the cursor position forward and stops at `LIMIT`. There is no scan.

```sql
-- 0020_pht_keyset.sql
CREATE INDEX ix_photos_taken_at_photo_desc
    ON photos (taken_at DESC, photo_id DESC);
```

:::{.callout-note}
The same index serves the timeline view (`GET /timeline/` in notebook 12) — the `date_trunc('month', taken_at)` grouping reads the first row of each month via the same index seek. We are not adding an index per feature; we are adding one index that serves two features.

:::


## The Response Envelope

Pagination responses always tell the client three things: the items, whether there are more items, and the cursor to use for the next request. We standardize on this envelope across all paginated endpoints (timeline, browse, search results, people album), so the client's infinite-scroll helper works unchanged against every list.


In [ ]:
from typing import Generic, TypeVar
from pydantic import BaseModel


T = TypeVar("T")


class Page(BaseModel, Generic[T]):
    items:       list[T]
    next_cursor: str | None = None
    has_more:    bool = False


# For browse specifically, the item type carries the derivative URLs from PHT:02.
class BrowseItem(BaseModel):
    photo_id:  str
    taken_at:  datetime
    urls:      dict[str, str | None]   # {"small": "...", "medium": "...", "large": "...", "original": None}


def serialize_browse_page(rows: list[dict], urls_map: dict[str, dict]) -> Page[BrowseItem]:
    items: list[BrowseItem] = []
    for row in rows:
        items.append(BrowseItem(
            photo_id=row["photo_id"],
            taken_at=row["taken_at"],
            urls=urls_map.get(row["photo_id"], {}),
        ))
    next_cursor = None
    has_more = False
    if rows:
        last = rows[-1]
        # In production this comes from fetch_page_keyset; here we just mark has_more=False.
        has_more = False
    return Page[BrowseItem](items=items, next_cursor=next_cursor, has_more=has_more)


# A demo page.
demo_rows = [
    {"photo_id": "p-100", "taken_at": datetime(2023, 8, 14, 17, 42, 0)},
    {"photo_id": "p-099", "taken_at": datetime(2023, 8, 13, 12, 0, 0)},
]
demo_urls = {
    "p-100": {"small": "https://s3/p100/small.jpg", "medium": "https://s3/p100/med.jpg",
              "large": "https://s3/p100/large.jpg", "original": None},
    "p-099": {"small": "https://s3/p099/small.jpg", "medium": "https://s3/p099/med.jpg",
              "large": "https://s3/p099/large.jpg", "original": None},
}
page = serialize_browse_page(demo_rows, demo_urls)
print(page.model_dump_json(indent=2))


## Presigned-URL Caching

The second cost dominates when the cache is cold: 30 presigned-URL generations per page, each a synchronous boto3 call into S3's signing code path (in production this is local — the signing key never leaves the host — but boto3's signing still does measurable work in a tight loop).

The insight: **a presigned URL is time-stable**. S3 accepts the URL until its expiry timestamp, so two identical `(bucket, key, expiry_window)` inputs produce two URLs that are functionally identical (differ only in the canonical request signature, which is deterministic). We can cache the URL itself and reuse it across requests until it is close to expiry.

The cache TTL must be **below** S3's expiry window by a safety margin. If we presign for 3600 s and cache the URL for 3500 s, the client gets a URL with at least 100 s of usability left — long enough for the next request to start before this one expires even over a slow network. If the cache served a URL with 50 s of life remaining and the client was on a 60 s latency connection, the URL would die mid-fetch. The 100 s margin is empirical.


In [ ]:
import time
import hashlib

try:
    import redis.asyncio as aioredis
    _HAS_REDIS = True
except ImportError:
    _HAS_REDIS = False


# A tiny async dict-cache stand-in for Redis, so the notebook runs without one.
class _InMemoryRedis:
    def __init__(self) -> None:
        self._store: dict[str, tuple[str, float]] = {}     # key -> (value, expires_at)

    async def get(self, key: str) -> str | None:
        v, exp = self._store.get(key, (None, 0.0))
        if v is None or exp < time.time():
            return None
        return v

    async def set(self, key: str, value: str, ex: int) -> None:
        self._store[key] = (value, time.time() + ex)

    async def delete(self, key: str) -> None:
        self._store.pop(key, None)


_PRESIGN_EXPIRY_S = 3600           # S3 URL validity
_PRESIGN_CACHE_TTL = 3500          # cache below expiry by 100 s
_CACHE_SAFETY_MARGIN = 100          # clients must get URLs with >= 100 s validity left


def _cache_key(bucket: str, key: str, expiry_s: int) -> str:
    """Cache key hashes (bucket, key, expiry_s). Two presign calls with the same
    inputs produce the same cache entry; differing expiry_s produces different
    entries, which is correct (URLs with different lives are not interchangeable)."""
    h = hashlib.sha256(f"{bucket}|{key}|{expiry_s}".encode("utf-8")).hexdigest()[:16]
    return f"presign:get:{h}"


async def cached_presign_get(
    s3_client,
    cache: _InMemoryRedis,
    bucket: str,
    key: str,
    expiry_s: int = _PRESIGN_EXPIRY_S,
) -> str:
    """Return a presigned GET URL, served from cache if recent."""
    ck = _cache_key(bucket, key, expiry_s)
    cached = await cache.get(ck)
    if cached is not None:
        return cached
    url = s3_client.generate_presigned_url(
        "get_object",
        Params={"Bucket": bucket, "Key": key},
        ExpiresIn=expiry_s,
    )
    await cache.set(ck, url, ex=_PRESIGN_CACHE_TTL)
    return url


# Demo: same key requested twice → second hit is served from cache.
from unittest.mock import MagicMock

mock_s3 = MagicMock(make_presigned=0)
mock_s3.generate_presigned_url = MagicMock(
    side_effect=lambda op, Params, ExpiresIn:
        f"https://{Params['Bucket']}.s3.example.com/{Params['Key']}?GET&exp={ExpiresIn}"
)
cache = _InMemoryRedis()

u1 = await cached_presign_get(mock_s3, cache, "my-photo-app-thumbnails", "thumbnails/aa/aa/small.jpg")
u2 = await cached_presign_get(mock_s3, cache, "my-photo-app-thumbnails", "thumbnails/aa/aa/small.jpg")

print(f"first call:  {u1[:60]}...")
print(f"second call: {u2[:60]}...")
print(f"identical URLs: {u1 == u2}")
print(f"generate_presigned_url call count: {mock_s3.generate_presigned_url.call_count}  (expected 1)")


## Cache Invalidation on Delete

When a photo is deleted (PHT:06), its presigned URLs become dangling references. The cache must be purged for `(original_bucket, original_key)` and every `(derivative_bucket, derivative_key)` tuple. We do this with a single `DEL` of each cache entry — small batches in practice (one original + up to three derivatives per photo).

:::{.callout-note}
**Why not cache by photo_id simpler?** Because presigning is per-(bucket, key) and our cache key includes `expiry_s`. Two cache keys (one for the original `expiry_s=3600`, one for derivatives with `expiry_s=3600`) coexist; deleting by `photo_id` would require a secondary index, and the savings (one `DEL` instead of four `DEL`s per photo) are not worth the Redis complexity.

:::


In [ ]:
async def invalidate_photo_urls(
    cache: _InMemoryRedis,
    s3_keys: list[str],
    buckets: list[str],
    expiry_s: int = _PRESIGN_EXPIRY_S,
) -> None:
    """Purge every cache entry that might serve a presigned URL for this photo's
    s3_keys across the listed buckets (original + derivative)."""
    for b in buckets:
        for key in s3_keys:
            await cache.delete(_cache_key(b, key, expiry_s))


# Demonstrate: cache a URL, invalidate, request again — should be a fresh URL.
mock_s3 = MagicMock()
mock_s3.generate_presigned_url = MagicMock(
    side_effect=lambda op, Params, ExpiresIn:
        f"https://{Params['Bucket']}.s3.example.com/{Params['Key']}?GET&exp={ExpiresIn}&nonce={time.time()}"
)
cache = _InMemoryRedis()
bucket, key = "my-photos", "photos/aa/abcd.jpg"

u1 = await cached_presign_get(mock_s3, cache, bucket, key)
await invalidate_photo_urls(cache, [key], [bucket])
u2 = await cached_presign_get(mock_s3, cache, bucket, key)

print(f"before invalidate: {u1[:60]}")
print(f"after invalidate:  {u2[:60]}")
print(f"URLs differ after invalidation: {u1 != u2}")
print(f"call count: 2 (re-presigned after invalidate)")


## The Browse Handler, End to End

Pulling the pieces together below — keyset query, serialize-to-`BrowseItem`, build-URLs-via-cache — we end up with a single async handler that is constant-cost at scroll depth. The handler does three I/Os: one DB seek, one cache read for each derivative URL (mostly hits), and one JSON serialize. Cache misses fall through to boto3 synchronously and the slow pages are the cold ones, which the previous section's `Cache-Control: immutable` on the S3 object heads means the *next* scroll session will likely be all hits.


In [ ]:
DERIVATIVE_BUCKET = "my-photo-app-thumbnails"
ORIGINAL_BUCKET    = "my-photos"


async def browse_handler(
    session: AsyncSession,
    s3_client,
    cache: _InMemoryRedis,
    page_size: int = 30,
    cursor: str | None = None,
    include_original: bool = False,
) -> Page[BrowseItem]:
    rows, next_cursor = await fetch_page_keyset(session, page_size, cursor)

    # Bulk-resolve URLs via cache. We do them concurrently with asyncio.gather
    # in the production version; here we sequence for clarity.
    urls_map: dict[str, dict] = {}
    for row in rows:
        # In production, fetch row's derivative keys from photo_derivatives (PHT:02).
        # Stub: assume all three derivatives exist with hash-stable keys.
        sha = row.get("sha256", "0000")
        deriv_keys = {
            "small":  f"thumbnails/{sha[:2]}/{sha}/small.jpg",
            "medium": f"thumbnails/{sha[:2]}/{sha}/medium.jpg",
            "large":  f"thumbnails/{sha[:2]}/{sha}/large.jpg",
        }
        urls = {}
        for size_name, key in deriv_keys.items():
            urls[size_name] = await cached_presign_get(s3_client, cache, DERIVATIVE_BUCKET, key)
        if include_original and row.get("s3_key"):
            urls["original"] = await cached_presign_get(s3_client, cache, ORIGINAL_BUCKET, row["s3_key"])
        urls_map[row["photo_id"]] = urls

    return serialize_browse_page(rows, urls_map)


# Demonstrate the data flow with stubbed helpers returning empty list.
cache = _InMemoryRedis()
s3 = MagicMock()
s3.generate_presigned_url = MagicMock(
    side_effect=lambda op, Params, ExpiresIn: f"https://{Params['Bucket']}/{Params['Key']}"
)
page = await browse_handler(session=MagicMock(), s3_client=s3, cache=cache)
print(f"browse handler returned {len(page.items)} items (stubbed empty)")


## The Infinite-Scroll Contract

The client keeps a small buffer of unrendered pages and prefetches the next one when the scroll position approaches the bottom. We pin the contract — what the client guarantees — into the API.

- The client sends `?cursor=<opaque>` on every page request after the first. The first page sends no `cursor`.
- The client treats `next_cursor=null` as the end of the list and **must** stop fetching.
- The client treats a 404 on `next_cursor` as "cursor too old, restart from initial" (we never age cursors, but defensive behavior is required).
- The client must not construct cursors client-side. Cursors are opaque; one client's cursor may not work for another client's session if we extend the cursor in a future notebook.

:::{.callout-important}
It is a contract violation to issue two concurrent `cursor=...` requests with the **same** cursor and interleave their responses: doing so causes duplicate photos in the rendered list. If the user scrolls past the buffer in one big jump, the client must wait for the active request to settle before issuing the next. The Flet `ListView` in the existing app already serializes scroll-driven prefetch, so this contract is honored for free.

:::


## Schema Additions

Keyset paging requires a single composite index that notebook 12 did not have. The presigned-URL cache lives entirely in Redis and adds nothing to the Postgres schema.

```sql
-- 0020_pht_keyset.sql
CREATE INDEX IF NOT EXISTS ix_photos_taken_at_photo_desc
    ON photos (taken_at DESC, photo_id DESC);
```

The `next_cursor` schema is part of the API contract, not a Postgres column. No changes to `photos`.


## Summary

This notebook replaced notebook 12's offset-paged, regenerate-the-presign browse path with a keyset-paged, cache-served browse path. Page render time at 100K photos is now bounded by one database seek and zero S3 presign calls on warm cache, neither of which grow with library size. The contract between client and server is now a documented infinite-scroll-with-cursor contract that the existing Flet `ListView` satisfies out of the box.

**What changed relative to notebook 12.** `GET /photos/?page=N` is replaced by `GET /photos/?cursor=...`. The response loses `presigned_url: str` and gains `urls: {small, medium, large, original}` plus `next_cursor` and `has_more`. A Redis cache sits behind the presign helper. A composite descending index joins the schema; the timeline query and the browse query share it.

**What this enables for the rest of PHT.** PHT:04's search response reuses the same `Page` envelope. PHT:06's delete cascade calls the cache invalidation helper. PHT:07's metrics include cache hit ratio per endpoint, which depends on this notebook's cache layer existing.

---



---


■
